# Loan Default Risk Analysis — Python Validation

This notebook validates key findings from the SQL analysis using Python: correlation analysis, a chi-square test of statistical significance, and a multivariate logistic regression model.

Dataset: Kaggle "Loan Default Prediction Dataset" (255,347 rows, 18 columns)

## Setup and Data Loading

Load the dataset and confirm it matches the same 255,347 rows and 18 columns used in the SQL analysis.

In [4]:
import os
print(os.getcwd())

/Users/rashawn


In [5]:
print(os.listdir())

['.config', 'Music', '.zprofile.pysave', 'Untitled1.ipynb', '.DS_Store', 'untitled1.py', '.CFUserTextEncoding', '.xonshrc', 'anaconda_projects', 'Untitled.ipynb', '.zshrc', 'Pictures', '.zprofile', '.zsh_history', 'Untitled2.ipynb', '.ipython', 'Desktop', 'Library', '.matplotlib', '.cups', 'Public', '.idlerc', '.tcshrc', '.anaconda', 'Movies', 'Applications', '.Trash', '.ipynb_checkpoints', '.jupyter', 'Documents', '.bash_profile', 'Downloads', '.python_history', '.continuum', '.gitconfig', 'Rashawn Hunt Resume 25.png', 'untitled.py', '.zsh_sessions', '.conda']


In [6]:
df = pd.read_csv('/Users/rashawn/Downloads/Loan_default.csv')
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [7]:
df.shape

(255347, 18)

## Correlation Check

SQL measured predictor strength using spread (the gap between the highest and lowest default rate across groups). Correlation gives a more formal, single-number version of the same idea for continuous variables, a value from -1 to +1 showing how strongly two variables move together.

This checks whether CreditScore, DTIRatio, and InterestRate line up with the same predictor ranking found in SQL.

In [8]:
df[['CreditScore', 'DTIRatio', 'InterestRate', 'Default']].corr()

,CreditScore,DTIRatio,InterestRate,Default
CreditScore,1.000000,-0.001039,0.000436,-0.034166
DTIRatio,-0.001039,1.000000,0.000575,0.019236
InterestRate,0.000436,0.000575,1.000000,0.131273
Default,-0.034166,0.019236,0.131273,1.000000


## Employment Type and Statistical Significance

Correlation only works on numeric columns, so a different approach is needed to test whether a categorical column like EmploymentType has a real relationship with Default. This section first confirms the default rate per employment type matches SQL, then formally tests significance using a chi-square test.

In [9]:
df.groupby('EmploymentType')['Default'].mean() * 100

EmploymentType
Full-time         9.463366
Part-time        11.965213
Self-employed    11.462029
Unemployed       13.552895
Name: Default, dtype: float64

In [10]:
contingency_table = pd.crosstab(df['EmploymentType'], df['Default'])
contingency_table

Default,0,1
EmploymentType,,
Full-time,57632,6024
Part-time,56484,7677
Self-employed,56404,7302
Unemployed,55174,8650


In [11]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("P-value:", p_value)

Chi-square statistic: 529.7449284156027
P-value: 1.7066378020433157e-114


## Multivariate Logistic Regression

Every check so far looked at one factor at a time. This section tests several factors together, holding others constant, using a logistic regression model. Since the model only understands numbers, the categorical EmploymentType column is first converted into numeric 0/1 columns using one-hot encoding.

In [12]:
pd.get_dummies(df['EmploymentType']).head()

,Full-time,Part-time,Self-employed,Unemployed
0,True,False,False,False
1,True,False,False,False
2,False,False,False,True
3,True,False,False,False
4,False,False,False,True


In [13]:
X = df[['CreditScore', 'DTIRatio', 'InterestRate']].copy()
X = pd.concat([X, pd.get_dummies(df['EmploymentType'], drop_first=True)], axis=1)
y = df['Default']

X.head()

,CreditScore,DTIRatio,InterestRate,Part-time,Self-employed,Unemployed
0,520,0.44,15.23,False,False,False
1,458,0.68,4.81,False,False,False
2,451,0.31,21.17,False,False,True
3,743,0.23,7.07,False,False,False
4,633,0.73,6.51,False,False,True


In [14]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X, y)

coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})
coefficients

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,feature,coefficient
0,CreditScore,-0.000701
1,DTIRatio,0.181487
2,InterestRate,0.063839
3,Part-time,0.320377
4,Self-employed,0.246234
5,Unemployed,0.449910


### Feature Scaling and Model Fit

CreditScore, DTIRatio, and InterestRate sit on very different numeric scales, which can prevent the model from converging properly and makes raw coefficients impossible to compare. StandardScaler rescales each to "standard deviations from the mean" first, so the resulting coefficients are directly comparable in size.

In [15]:
from sklearn.preprocessing import StandardScaler

X_scaled = X.copy()
numeric_cols = ['CreditScore', 'DTIRatio', 'InterestRate']
scaler = StandardScaler()
X_scaled[numeric_cols] = scaler.fit_transform(X_scaled[numeric_cols])

model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

coefficients = pd.DataFrame({
    'feature': X_scaled.columns,
    'coefficient': model.coef_[0]
})
coefficients

,feature,coefficient
0,CreditScore,-0.109684
1,DTIRatio,0.061242
2,InterestRate,0.423690
3,Part-time,0.269802
4,Self-employed,0.220410
5,Unemployed,0.414373


## Summary

All three Python checks (correlation, chi-square significance test, and multivariate logistic regression) confirm the same predictor ranking found in the SQL analysis: Interest Rate and Employment Status are the strongest signals, DTI Ratio is the weakest. The chi-square test confirms EmploymentType's relationship with Default is statistically significant (p-value near zero), and the logistic regression confirms this holds even when controlling for other factors simultaneously.